<a href="https://colab.research.google.com/github/prasad0876/231FA04140-MLOps-Feast-SkillGap/blob/main/MLOPS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pandas numpy scikit-learn joblib pyarrow feast==0.64.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.1/64.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires tenacity<10

In [2]:
import os
import shutil
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import feast
from feast import FeatureStore

print("Feast version:", feast.__version__)


Feast version: 0.64.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
from google.colab import files

uploaded = files.upload()

data = pd.read_csv("/content/CSE_Employability_Skill_Dataset_5000.csv")

print("Dataset shape:", data.shape)
display(data.head())


Saving CSE_Employability_Skill_Dataset_5000.xlsx to CSE_Employability_Skill_Dataset_5000 (1).xlsx
Dataset shape: (5000, 13)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,Student_ID,Programming,Databases,Problem_Solving,Communication,Cloud_Computing,Teamwork,Aptitude,Data_Analysis,Average_Skill_Gap,Skill_Gap_Category,Overall_Skill_Score,Training_Recommendation
0,STU00001,75.5,58.2,57.5,61.8,64.3,72.2,36.3,76.5,13.40,Medium,62.79,Aptitude
1,STU00002,65.9,57.7,62.7,63.5,63.1,70.2,50.2,88.6,12.46,Medium,65.24,Aptitude
2,STU00003,77.7,36.3,58.6,65.0,41.1,64.4,57.2,66.9,17.22,Medium,58.40,Databases
3,STU00004,90.8,59.7,68.5,78.2,68.4,70.0,68.2,56.7,7.31,Low,70.06,Databases
4,STU00005,64.5,76.7,83.8,52.8,31.2,76.4,81.4,42.9,13.20,Medium,63.71,Cloud_Computing


In [4]:
print("Shape:", data.shape)
print("\nColumns:")
print(data.columns.tolist())

print("\nData types:")
print(data.dtypes)

print("\nMissing values:")
print(data.isnull().sum())

print("\nDuplicate rows:", data.duplicated().sum())
print("Duplicate Student IDs:", data["Student_ID"].duplicated().sum())


Shape: (5000, 13)

Columns:
['Student_ID', 'Programming', 'Databases', 'Problem_Solving', 'Communication', 'Cloud_Computing', 'Teamwork', 'Aptitude', 'Data_Analysis', 'Average_Skill_Gap', 'Skill_Gap_Category', 'Overall_Skill_Score', 'Training_Recommendation']

Data types:
Student_ID                  object
Programming                float64
Databases                  float64
Problem_Solving            float64
Communication              float64
Cloud_Computing            float64
Teamwork                   float64
Aptitude                   float64
Data_Analysis              float64
Average_Skill_Gap          float64
Skill_Gap_Category          object
Overall_Skill_Score        float64
Training_Recommendation     object
dtype: object

Missing values:
Student_ID                 0
Programming                0
Databases                  0
Problem_Solving            0
Communication              0
Cloud_Computing            0
Teamwork                   0
Aptitude                   0
Data_Anal

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:
data = data.drop_duplicates().copy()
data = data.drop_duplicates(subset=["Student_ID"], keep="first").copy()

for col in ["Skill_Gap_Category", "Training_Recommendation"]:
    data[col] = data[col].astype(str).str.strip()

print("Cleaned dataset shape:", data.shape)

Cleaned dataset shape: (5000, 13)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:
target = "Skill_Gap_Category"

excluded = {
    "Student_ID",
    target,
    "Overall_Gap_Score",
    "Priority_Training_Area",
    "Employability_Readiness"
}

# Exclude all individual calculated gap columns.
excluded.update([
    c for c in data.columns
    if c.endswith("_Gap")
])

feature_cols = [
    c for c in data.columns
    if c not in excluded
]

X = data[feature_cols].copy()
y = data[target].copy()

print("Number of input features:", len(feature_cols))
print("\nInput features:")
for col in feature_cols:
    print("-", col)

print("\nTarget distribution:")
print(y.value_counts())

Number of input features: 10

Input features:
- Programming
- Databases
- Problem_Solving
- Communication
- Cloud_Computing
- Teamwork
- Aptitude
- Data_Analysis
- Overall_Skill_Score
- Training_Recommendation

Target distribution:
Skill_Gap_Category
Medium    3439
Low       1258
High       303
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [7]:
categorical_features = [
    c for c in feature_cols
    if X[c].dtype == "object"
]

numerical_features = [
    c for c in feature_cols
    if c not in categorical_features
]

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['Programming', 'Databases', 'Problem_Solving', 'Communication', 'Cloud_Computing', 'Teamwork', 'Aptitude', 'Data_Analysis', 'Overall_Skill_Score']

Categorical features:
['Training_Recommendation']


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 4000
Testing records: 1000


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [9]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [10]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

In [11]:
preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])

In [12]:
ml_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

ml_pipeline.fit(X_train, y_train)

predictions = ml_pipeline.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", round(accuracy * 100, 2), "%")
print("\nClassification Report:")
print(classification_report(y_test, predictions))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  opt_res = optimize.minimize(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Accuracy: 89.7 %

Classification Report:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


              precision    recall  f1-score   support

        High       0.94      0.78      0.85        60
         Low       0.83      0.83      0.83       252
      Medium       0.92      0.93      0.93       688

    accuracy                           0.90      1000
   macro avg       0.90      0.85      0.87      1000
weighted avg       0.90      0.90      0.90      1000



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [13]:
cm = confusion_matrix(y_test, predictions)

cm_df = pd.DataFrame(
    cm,
    index=ml_pipeline.classes_,
    columns=ml_pipeline.classes_
)

# Rename index and columns for better readability
cm_df.index.name = "Actual"
cm_df.columns.name = "Predicted"

display(cm_df)

Predicted,High,Low,Medium
Actual,,,
High,47,0,13
Low,0,208,44
Medium,3,43,642


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [14]:
X_train_processed = ml_pipeline.named_steps[
    "preprocessing"
].transform(X_train)

X_test_processed = ml_pipeline.named_steps[
    "preprocessing"
].transform(X_test)

processed_feature_names = ml_pipeline.named_steps[
    "preprocessing"
].get_feature_names_out()

processed_train = pd.DataFrame(
    X_train_processed,
    columns=processed_feature_names,
    index=X_train.index
)

processed_test = pd.DataFrame(
    X_test_processed,
    columns=processed_feature_names,
    index=X_test.index
)

processed_train[target] = y_train.values
processed_test[target] = y_test.values

print("Processed training shape:", processed_train.shape)
print("Processed testing shape:", processed_test.shape)

display(processed_train.head())

Processed training shape: (4000, 18)
Processed testing shape: (1000, 18)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,numerical__Programming,numerical__Databases,numerical__Problem_Solving,numerical__Communication,numerical__Cloud_Computing,numerical__Teamwork,numerical__Aptitude,numerical__Data_Analysis,numerical__Overall_Skill_Score,categorical__Training_Recommendation_Aptitude,categorical__Training_Recommendation_Cloud_Computing,categorical__Training_Recommendation_Communication,categorical__Training_Recommendation_Data_Analysis,categorical__Training_Recommendation_Databases,categorical__Training_Recommendation_Problem_Solving,categorical__Training_Recommendation_Programming,categorical__Training_Recommendation_Teamwork,Skill_Gap_Category
3824,-0.697284,0.902341,2.058784,0.737957,0.888614,-0.496087,-0.629767,0.362775,1.158084,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,Low
323,2.137911,0.228399,-0.292134,0.443022,-1.155362,-0.201401,0.763048,-1.761442,-0.066768,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,Medium
2788,-0.471834,-0.042438,-0.998852,-1.192530,-0.139083,0.132060,0.459401,-0.238075,-0.868284,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,Medium
4074,-0.465003,2.086464,2.275126,-1.279671,-0.527324,-1.131990,0.195360,-0.165244,0.356568,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,Medium
3968,2.055929,0.549624,-1.006063,-0.897595,-0.841343,-0.519352,0.393391,-1.069554,-0.508919,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,Medium


In [15]:
processed_train.to_csv(
    "CSE_employability_processed_train.csv",
    index=False
)

processed_test.to_csv(
    "CSE_employability_processed_test.csv",
    index=False
)

joblib.dump(
    ml_pipeline,
    "CSE_employability_preprocessing_model_pipeline.pkl"
)

print("Preprocessed files and ML pipeline saved.")

Preprocessed files and ML pipeline saved.


In [16]:
# Create Feast-compatible feature data.
feast_df = data.copy()

feast_df["student_id"] = feast_df["Student_ID"].astype(str)

# Deterministic timestamps for the feature-store demonstration.
base_time = pd.Timestamp("2026-01-01", tz="UTC")

feast_df["event_timestamp"] = (
    base_time +
    pd.to_timedelta(
        np.arange(len(feast_df)),
        unit="s"
    )
)

feast_df["created_timestamp"] = (
    feast_df["event_timestamp"] +
    pd.Timedelta(seconds=1)
)

# Features used by Feast.
# Corrected to use available columns from the 'data' DataFrame.
feast_feature_columns = [
    "student_id",
    "event_timestamp",
    "created_timestamp",
    "Programming",
    "Databases",
    "Problem_Solving",
    "Communication",
    "Cloud_Computing",
    "Teamwork",
    "Aptitude",
    "Data_Analysis",
    "Overall_Skill_Score",
    "Training_Recommendation"
]

feast_features = feast_df[feast_feature_columns].copy()

display(feast_features.head())

,student_id,event_timestamp,created_timestamp,Programming,Databases,Problem_Solving,Communication,Cloud_Computing,Teamwork,Aptitude,Data_Analysis,Overall_Skill_Score,Training_Recommendation
0,STU00001,2026-01-01 00:00:00+00:00,2026-01-01 00:00:01+00:00,75.5,58.2,57.5,61.8,64.3,72.2,36.3,76.5,62.79,Aptitude
1,STU00002,2026-01-01 00:00:01+00:00,2026-01-01 00:00:02+00:00,65.9,57.7,62.7,63.5,63.1,70.2,50.2,88.6,65.24,Aptitude
2,STU00003,2026-01-01 00:00:02+00:00,2026-01-01 00:00:03+00:00,77.7,36.3,58.6,65.0,41.1,64.4,57.2,66.9,58.40,Databases
3,STU00004,2026-01-01 00:00:03+00:00,2026-01-01 00:00:04+00:00,90.8,59.7,68.5,78.2,68.4,70.0,68.2,56.7,70.06,Databases
4,STU00005,2026-01-01 00:00:04+00:00,2026-01-01 00:00:05+00:00,64.5,76.7,83.8,52.8,31.2,76.4,81.4,42.9,63.71,Cloud_Computing


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [17]:
repo_path = "/content/cse_employability_feast"

# Recreate repository for a clean run.
if os.path.exists(repo_path):
    shutil.rmtree(repo_path)

os.makedirs(f"{repo_path}/data", exist_ok=True)

feast_features.to_parquet(
    f"{repo_path}/data/student_features.parquet",
    index=False
)

print("Feast repository created at:", repo_path)

Feast repository created at: /content/cse_employability_feast


In [18]:
feature_store_config = '''
project: cse_employability_project

registry: data/registry.db

provider: local

offline_store:
  type: file

online_store:
  type: sqlite
  path: data/online_store.db
'''

with open(
    f"{repo_path}/feature_store.yaml",
    "w"
) as f:
    f.write(feature_store_config)

print("feature_store.yaml created.")

feature_store.yaml created.


In [19]:
feature_definition = '''
from datetime import timedelta

from feast import (
    Entity,
    FeatureView,
    FeatureService,
    Field,
    FileSource
)

from feast.types import Float32, Int64, String

student = Entity(
    name="student",
    join_keys=["student_id"],
    description="CSE graduate student"
)

student_source = FileSource(
    name="student_skill_source",
    path="data/student_features.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp"
)

student_skill_features = FeatureView(
    name="student_skill_features",
    entities=[student],
    ttl=timedelta(days=3650),
    schema=[
        Field(name="Programming", dtype=Float32),
        Field(name="Databases", dtype=Float32),
        Field(name="Problem_Solving", dtype=Float32),
        Field(name="Communication", dtype=Float32),
        Field(name="Cloud_Computing", dtype=Float32),
        Field(name="Teamwork", dtype=Float32),
        Field(name="Aptitude", dtype=Float32),
        Field(name="Data_Analysis", dtype=Float32),
        Field(name="Overall_Skill_Score", dtype=Float32),
        Field(name="Training_Recommendation", dtype=String),
    ],
    source=student_source,
    online=True
)

employability_service = FeatureService(
    name="cse_employability_service",
    features=[student_skill_features]
)
'''

with open(
    f"{repo_path}/features.py",
    "w"
) as f:
    f.write(feature_definition)

print("Feast feature definitions created.")

Feast feature definitions created.


In [20]:
%cd /content/cse_employability_feast

!feast apply


/content/cse_employability_feast
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/

In [21]:
!feast entities list
!feast feature-views list


/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [22]:
store = FeatureStore(
    repo_path=repo_path
)

feature_service = store.get_feature_service(
    "cse_employability_service"
)

print("Feast Feature Store ready.")


Feast Feature Store ready.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [23]:
entity_df = feast_df[
    ["student_id", "event_timestamp"]
].copy()

historical_features = store.get_historical_features(
    entity_df=entity_df,
    features=feature_service
).to_df()

print("Historical feature shape:", historical_features.shape)
display(historical_features.head())


Historical feature shape: (5000, 12)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,student_id,event_timestamp,Programming,Databases,Problem_Solving,Communication,Cloud_Computing,Teamwork,Aptitude,Data_Analysis,Overall_Skill_Score,Training_Recommendation
0,STU00001,2026-01-01 00:00:00+00:00,75.5,58.2,57.5,61.8,64.3,72.2,36.3,76.5,62.79,Aptitude
1,STU00002,2026-01-01 00:00:01+00:00,65.9,57.7,62.7,63.5,63.1,70.2,50.2,88.6,65.24,Aptitude
2,STU00003,2026-01-01 00:00:02+00:00,77.7,36.3,58.6,65.0,41.1,64.4,57.2,66.9,58.40,Databases
3,STU00004,2026-01-01 00:00:03+00:00,90.8,59.7,68.5,78.2,68.4,70.0,68.2,56.7,70.06,Databases
4,STU00005,2026-01-01 00:00:04+00:00,64.5,76.7,83.8,52.8,31.2,76.4,81.4,42.9,63.71,Cloud_Computing


In [24]:
%cd /content/cse_employability_feast

start_time = feast_features["event_timestamp"].min().strftime(
    "%Y-%m-%dT%H:%M:%S"
)

end_time = (
    feast_features["event_timestamp"].max()
    + pd.Timedelta(minutes=1)
).strftime("%Y-%m-%dT%H:%M:%S")

print("Materializing from:", start_time)
print("Materializing to:", end_time)

!feast materialize $start_time $end_time


/content/cse_employability_feast
Materializing from: 2026-01-01T00:00:00
Materializing to: 2026-01-01T01:24:19
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 

In [25]:
sample_student_id = feast_features.iloc[0]["student_id"]

online_features = store.get_online_features(
    features=feature_service,
    entity_rows=[
        {"student_id": sample_student_id}
    ]
).to_dict()

online_df = pd.DataFrame(online_features)

display(online_df)


,student_id,Communication,Databases,Teamwork,Data_Analysis,Cloud_Computing,Overall_Skill_Score,Training_Recommendation,Programming,Aptitude,Problem_Solving
0,STU00001,61.799999,58.200001,72.199997,76.5,64.300003,62.790001,Aptitude,75.5,36.299999,57.5


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [26]:
# Build a prediction row from the original feature values.
prediction_row = feast_df[
    feast_df["student_id"] == sample_student_id
][feature_cols].copy()

prediction = ml_pipeline.predict(prediction_row)

print("Student ID:", sample_student_id)
print("Predicted Skill-Gap Category:", prediction[0])


Student ID: STU00001
Predicted Skill-Gap Category: Medium


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [27]:
sample_ids = feast_features.iloc[
    :5
]["student_id"].tolist()

multi_online = store.get_online_features(
    features=feature_service,
    entity_rows=[
        {"student_id": sid}
        for sid in sample_ids
    ]
).to_dict()

multi_online_df = pd.DataFrame(multi_online)

display(multi_online_df)


,student_id,Communication,Databases,Teamwork,Data_Analysis,Cloud_Computing,Overall_Skill_Score,Training_Recommendation,Programming,Aptitude,Problem_Solving
0,STU00001,61.799999,58.200001,72.199997,76.500000,64.300003,62.790001,Aptitude,75.500000,36.299999,57.500000
1,STU00002,63.500000,57.700001,70.199997,88.599998,63.099998,65.239998,Aptitude,65.900002,50.200001,62.700001
2,STU00003,65.000000,36.299999,64.400002,66.900002,41.099998,58.400002,Databases,77.699997,57.200001,58.599998
3,STU00004,78.199997,59.700001,70.000000,56.700001,68.400002,70.059998,Databases,90.800003,68.199997,68.500000
4,STU00005,52.799999,76.699997,76.400002,42.900002,31.200001,63.709999,Cloud_Computing,64.500000,81.400002,83.800003


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [28]:
 from google.colab import files

%cd /content/

files.download("CSE_employability_processed_train.csv")
files.download("CSE_employability_processed_test.csv")
files.download("CSE_employability_preprocessing_model_pipeline.pkl")

/content


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [33]:
from IPython.display import display, HTML, Markdown

# Helper to print professional headers
def print_header(text):
    display(HTML(f"""
    <div style='background-color: #f0f2f6; padding: 10px; border-radius: 5px; border-left: 5px solid #2e7d32; margin-top: 20px; margin-bottom: 10px;'>
        <h3 style='color: #2e7d32; margin: 0; font-family: sans-serif;'>{text}</h3>
    </div>
    """))

# 1. Historical Feature Output
print_header("1. Historical Feature Store Snapshot")
display(historical_features.head())

# 2. Model Performance
print_header("2. Model Evaluation")
accuracy = accuracy_score(y_test, predictions)
display(HTML(f"""
<div style='padding: 15px; border: 1px solid #ddd; border-radius: 5px; text-align: center; background-color: #fafafa;'>
    <span style='font-size: 1.2em; color: #555;'>Logistic Regression Accuracy:</span><br>
    <b style='font-size: 2.5em; color: #1a73e8;'>{accuracy * 100:.2f}%</b>
</div>
"""))

# 3. Online Feature Output
print_header("3. Online Feature Retrieval (Real-time)")
sample_sid = feast_features.iloc[0]["student_id"]
online_resp = store.get_online_features(
    features=feature_service,
    entity_rows=[{"student_id": sample_sid}]
).to_dict()
display(pd.DataFrame(online_resp))

# 4. Final Inference
print_header("4. Predictive Inference")
final_pred_row = feast_df[feast_df["student_id"] == sample_sid][feature_cols]
final_pred = ml_pipeline.predict(final_pred_row)

display(Markdown(f"""
| Entity Type | Entity ID | Predicted Employability Gap |
| :--- | :--- | :--- |
| **Student** | `{sample_sid}` | <span style='color: #d32f2f; font-weight: bold;'>{final_pred[0]}</span> |
"""))

,student_id,event_timestamp,Programming,Databases,Problem_Solving,Communication,Cloud_Computing,Teamwork,Aptitude,Data_Analysis,Overall_Skill_Score,Training_Recommendation
0,STU00001,2026-01-01 00:00:00+00:00,75.5,58.2,57.5,61.8,64.3,72.2,36.3,76.5,62.79,Aptitude
1,STU00002,2026-01-01 00:00:01+00:00,65.9,57.7,62.7,63.5,63.1,70.2,50.2,88.6,65.24,Aptitude
2,STU00003,2026-01-01 00:00:02+00:00,77.7,36.3,58.6,65.0,41.1,64.4,57.2,66.9,58.40,Databases
3,STU00004,2026-01-01 00:00:03+00:00,90.8,59.7,68.5,78.2,68.4,70.0,68.2,56.7,70.06,Databases
4,STU00005,2026-01-01 00:00:04+00:00,64.5,76.7,83.8,52.8,31.2,76.4,81.4,42.9,63.71,Cloud_Computing


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,student_id,Communication,Databases,Teamwork,Data_Analysis,Cloud_Computing,Overall_Skill_Score,Training_Recommendation,Programming,Aptitude,Problem_Solving
0,STU00001,61.799999,58.200001,72.199997,76.5,64.300003,62.790001,Aptitude,75.5,36.299999,57.5


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



| Entity Type | Entity ID | Predicted Employability Gap |
| :--- | :--- | :--- |
| **Student** | `STU00001` | <span style='color: #d32f2f; font-weight: bold;'>Medium</span> |


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
